In [1]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.zoo as foz
import fiftyone.brain.internal.models as fbm
import fiftyone.utils.random as four
fob.brain_config.default_visualization_method = "umap"

In [2]:
#fob.compute_hardness(samples, label_field, hardness_field='hardness', progress=None)

In [4]:

if fo.dataset_exists("quickstart"):
    fo.delete_dataset("quickstart")

#dataset = fo.load_dataset("malaria-cell-images")
dataset_zoo = foz.load_zoo_dataset("quickstart")
dataset_cifar = foz.load_zoo_dataset("cifar10", split="test")

Dataset already downloaded
Loading 'quickstart'
 100% |█████████████████| 200/200 [3.7s elapsed, 0s remaining, 49.8 samples/s]      
Dataset 'quickstart' created
Split 'test' already downloaded
Loading existing dataset 'cifar10-test'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [4]:
#using downloaded dataset as a main pre test for the particle dataset

In [5]:
"""
Model selection
"""

'\nModel selection\n'

In [9]:
model = fbm.load_model("simple-resnet-cifar10")
embeddings = dataset_cifar.compute_embeddings(model, batch_size=16)

 100% |█████████████| 10000/10000 [1.2m elapsed, 0s remaining, 146.2 samples/s]      


In [ ]:
#visualization test for using ground truth as a factor for comparison

In [12]:
# Generate a visualization with a difference based on their ground truth labels
results = fob.compute_visualization(
    dataset_cifar,
    brain_key="cifar_viz",
    create_index=True,
    embeddings=embeddings, #embedding must be exposed
    method="umap",
    min_dist=0.2,
    num_dims=2,
    #points_field=brain_key
    #patches_field="ground_truth" #for comparing the ground truth between different samples
)


Generating visualization...
UMAP(min_dist=0.2, verbose=True)
Mon Sep 15 16:59:38 2025 Construct fuzzy simplicial set
Mon Sep 15 16:59:38 2025 Finding Nearest Neighbors
Mon Sep 15 16:59:38 2025 Building RP forest with 10 trees
Mon Sep 15 16:59:44 2025 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	Stopping threshold met -- exiting after 4 iterations
Mon Sep 15 16:59:56 2025 Finished Nearest Neighbor Search
Mon Sep 15 16:59:58 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Mon Sep 15 17:00:04 2025 Finished embedding
Generating spatial index in field 'cifar_viz'...
 100% |█████████████| 10000/10000 [891.5ms elapsed, 0s remaining, 11.2K samples/s]      


In [6]:
print(results.has_spatial_index)
# True/False

# Add a spatial index to existing visualization results
results.index_points()

# Remove the spatial index from existing visualization results
results.remove_index()

True
Generating spatial index in field 'cifar_viz'...
 100% |█████████████| 10000/10000 [1.4s elapsed, 0s remaining, 7.2K samples/s]         


In [6]:
"""
Sort by near duplicate
"""

'\nSort by near duplicate\n'

In [7]:
near_duplicate = fob.compute_near_duplicates(
    dataset_cifar,
    threshold = 0.02, 
    embeddings=embeddings,
)
dups_view = near_duplicate.duplicates_view(
    type_field="dup_type",
    id_field="dup_id",
    dist_field="dup_dist",
)

Computing duplicate samples...


Traceback (most recent call last):
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 110: character maps to <undefined>


Duplicates computation complete


In [ ]:
"""
Uniqueness detection
"""

In [15]:
#detecting the highest 500 unique samples
near_duplicate.find_unique(500)
print(near_duplicate.unique_ids[5])

unique_view=dataset_cifar.select(near_duplicate.unique_ids)

Computing unique samples...
threshold: 1.000000, kept: 5, target: 500
threshold: 0.500000, kept: 1478, target: 500
threshold: 0.750000, kept: 106, target: 500
threshold: 0.625000, kept: 489, target: 500
threshold: 0.562500, kept: 875, target: 500
threshold: 0.593750, kept: 656, target: 500
threshold: 0.609375, kept: 561, target: 500
threshold: 0.617188, kept: 526, target: 500
threshold: 0.621094, kept: 513, target: 500
threshold: 0.623047, kept: 500, target: 500
Uniqueness computation complete
68c7c6fa04c6baf29b8e5e59


In [13]:
fob.compute_uniqueness(dataset_cifar)
#returns value from 0 to 1 which 1 meaning the image is more unique
#value can be observed using the launch app with the selected dataset

Computing embeddings...
 100% |█████████████| 10000/10000 [1.2m elapsed, 0s remaining, 143.9 samples/s]      
Computing uniqueness...
Uniqueness computation complete


In [ ]:
"""
Sample hardness/mistakes from label 
2 in 1
"""

In [13]:
fob.compute_mistakenness(dataset_cifar, "predictions", label_field="ground_truth")

ValueError: Dataset has no sample field 'predictions'

In [ ]:
"""
Leaky split configuration
"""

In [7]:
dataset_cifar.untag_samples(dataset_cifar.distinct("tags"))
four.random_split(dataset_cifar, {"train":0.7, "test":0.3})

#setup splits based on the different tags
split_tags = ["train","test"]
leaky_split_index = fob.compute_leaky_splits(dataset_cifar, splits=split_tags)
leaks = leaky_split_index.leaks_view()


Computing embeddings...
 100% |█████████████| 10000/10000 [6.3m elapsed, 0s remaining, 25.6 samples/s]      
Computing duplicate samples...


Traceback (most recent call last):
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 110: character maps to <undefined>


Duplicates computation complete


In [14]:
#DATASET FIFTYONE LAUNCHER
#session = fo.launch_app(leaks)#change the dataset into desired variable
#session = fo.launch_app(dataset_zoo) #for quickstart dataset
session = fo.launch_app(dataset_cifar) #for dataset from cifar
#session = fo.launch_app(dups_view) #for near duplicates
#session = fo.launch_app(view=unique_view) #for maximal uniqueness detection
session.wait()
url = session.url
print(f"Fiftyone link: {url}")

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port


Notebook sessions cannot wait
Fiftyone link: http://localhost:5151/


In [6]:
view = dataset.take(50)

with results.use_view(view):
    print(results.index_size)  # 50

    plot = results.visualize()
    plot.show()

50


FigureWidget({
    'data': [{'customdata': array(['68c7d2ec74a8d4af11f85256', '68c7d2f074a8d4af11f8530e',
                                   '68c7d2ec74a8d4af11f8524e', '68c7d2ec74a8d4af11f8526f',
                                   '68c7d2ed74a8d4af11f852a8', '68c7d2ef74a8d4af11f852fc',
                                   '68c7d2ef74a8d4af11f852f4', '68c7d2ed74a8d4af11f85288',
                                   '68c7d2ee74a8d4af11f852dd', '68c7d2f074a8d4af11f85306',
                                   '68c7d2ef74a8d4af11f852e8', '68c7d2ed74a8d4af11f8527f',
                                   '68c7d2f074a8d4af11f8530f', '68c7d2ec74a8d4af11f85276',
                                   '68c7d2ee74a8d4af11f852be', '68c7d2ee74a8d4af11f852b9',
                                   '68c7d2ed74a8d4af11f852aa', '68c7d2ed74a8d4af11f852ad',
                                   '68c7d2ec74a8d4af11f8527a', '68c7d2f074a8d4af11f8530c',
                                   '68c7d2f074a8d4af11f85303', '68c7d2ed74a

In [10]:
 #Convert to patches view
patches = dataset.to_patches("ground_truth")

# Choose a random patch object from the dataset
query_id = patches.take(1).first().id

# Programmatically construct a view containing the 15 most similar objects
view = patches.sort_by_similarity(query_id, k=15, brain_key="gt_sim")

session.view = view

view_cat = dataset.sort_by_similarity(query="a cat", k=15, brain_key="img_sim")
session.view = view_cat

In [ ]:
"""below here is the git init workspace
"""

In [7]:
!git init

Initialized empty Git repository in C:/Users/ERFIGO/Documents/fiftyoneproject-master/.git/


In [15]:
!git status

On branch branch1-embedding

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitattributes
	new file:   .gitignore
	new file:   .ipynb_checkpoints/fiftyone_embedding-checkpoint.ipynb
	new file:   fiftyone_embedding.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   fiftyone_embedding.ipynb



In [13]:
!git commit -m "feat: updating fiftyone embedding project"

On branch branch1-embedding
Your branch is up to date with 'fiftyone/branch1-embedding'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   fiftyone_embedding.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/

no changes added to commit (use "git add" and/or "git commit -a")


In [14]:
#below this is to update the .ipynb files or push those files into the branch

In [15]:
!git rm -r --cached .ipynb_checkpoints 2> NUL

In [16]:
!print ".ipynb_checkpoints/\n" >> .gitignore

In [17]:
!git add .gitignore

In [18]:
!git commit -m "Ignore Jupyter checkpoints"

[branch1-embedding 29eff86] Ignore Jupyter checkpoints
 1 file changed, 1 insertion(+)


In [19]:
!print "* text=auto\n*.ipynb -text\n" >> .gitattributes

In [20]:
!git add .gitattributes

In [21]:
!git commit -m "feat: tested bdd100k labels and images and ground truth with pytorch"

[branch1-embedding 0a0b5e5] feat: tested bdd100k labels and images and ground truth with pytorch
 1 file changed, 1 insertion(+)


In [22]:
!git push

To https://github.com/erfigo/fiftyoneproject.git
   5a8d4da..0a0b5e5  branch1-embedding -> branch1-embedding


In [34]:
#to add new remote origin from a new gpu

In [24]:
!git remote add fiftyone https://github.com/erfigo/fiftyoneproject.git

In [26]:
!git push

branch 'branch1-embedding' set up to track 'fiftyone/branch1-embedding'.


remote: 
remote: Create a pull request for 'branch1-embedding' on GitHub by visiting:        
remote:      https://github.com/erfigo/fiftyoneproject/pull/new/branch1-embedding        
remote: 
To https://github.com/erfigo/fiftyoneproject.git
 * [new branch]      branch1-embedding -> branch1-embedding


In [14]:
"""
ADD AND PUSH NEW UPDATES HERE TO THE SELECTED BRANCH
"""

'\nADD AND PUSH NEW UPDATES HERE TO THE SELECTED BRANCH\n'

In [11]:
!git checkout branch1-embedding

Switched to a new branch 'branch1-embedding'


In [17]:
!git add fiftyone_additional_features.ipynb .gitignore

In [18]:
!git commit -m "feat: adding uniqueness feature"

[branch1-embedding 048b7a4] feat: adding uniqueness feature
 1 file changed, 188 insertions(+), 35 deletions(-)


In [19]:
!git push

To https://github.com/erfigo/fiftyoneproject.git
   9bbdf7b..048b7a4  branch1-embedding -> branch1-embedding


In [3]:
!fiftyone zoo datasets info bdd100k

Der Befehl "fiftyone" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
